##SETUP

In [ ]:
!pip install gensim keybert altair dask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.8 MB/s eta 0:00:00


In [ ]:
!pip install datasets

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bahramjannesarr/goodreads-book-datasets-10m")

print("Path to dataset files:", path)

100%|██████████| 460M/460M [00:13<00:00, 35.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/bahramjannesarr/goodreads-book-datasets-10m/versions/18


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
alt.renderers.enable('mimetype')
from wordcloud import WordCloud
import re
import string
import pickle
#import for data cleaning of stopwords and converting words to base words
import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
from nltk.classify.textcat import TextCat
#import for converting each book tags into vector
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# Multiprocessing
import dask.dataframe as dd
import multiprocessing
# BERT-Embeddings
from keybert import KeyBERT

#DATA

In [ ]:
df=pd.read_csv(r'you can use my datasets that is already cleaned, check readme.md')

#EDA

##DATASET INFO

In [ ]:
print(df.info())

In [ ]:
print(df.shape)

##CHECK FOR MISSING VALUE

In [ ]:
print("Dataset Buku:\n", df.isnull().sum())

##CHECK FOR DUPLICATION

In [ ]:
print("Title duplicates:", df['title'].duplicated().sum())

In [ ]:
print("Author duplicates:", df['authors'].duplicated().sum())

In [ ]:
print("Genre duplicates:", df['categories'].duplicated().sum())

In [ ]:
print("Desc duplicates:", df['description'].duplicated().sum())

##TITLE AND DESC LENGTH

In [ ]:
df["name_length"] = df["title"].astype(str).apply(lambda x: len(x.split()))
print("name length stats:")
print(df["name_length"].describe())

In [ ]:
plt.figure(figsize=(8,5))
ax=sns.histplot(df["name_length"], bins=50, kde=False)
plt.title("title length distribution")
plt.xlabel("title words count")
plt.ylabel("Freq")
plt.show()

In [ ]:
df["desc_length"] = df["description"].astype(str).apply(lambda x: len(x.split()) if x != "nan" else 0)
print("description length stats:")
print(df["desc_length"].describe())

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["desc_length"], bins=50, kde=False)
plt.title("desc length distribution")
plt.xlabel("desc words count")
plt.ylabel("freq")
plt.show()

##TITLE ANALYSIS

In [ ]:
top_authors = df["title"].value_counts().head(10)
plt.figure(figsize=(10,6))
ax=sns.barplot(y=top_authors.index, x=top_authors.values, palette="viridis")
for i, v in enumerate(top_authors.values):
    ax.text(v + 3, i, str(v), color='black', va='center')
plt.title("title with duplicates")
plt.xlabel("number of books")
plt.ylabel("title")
plt.show()

##AUTHOR ANALYSIS

In [ ]:
print("number of unique authors:", df["authors"].nunique())

In [ ]:
top_authors = df["authors"].value_counts().head(10)
plt.figure(figsize=(10,6))
ax=sns.barplot(y=top_authors.index, x=top_authors.values, palette="viridis")
for i, v in enumerate(top_authors.values):
    ax.text(v + 3, i, str(v), color='black', va='center')
plt.title("Top 10 authors based on number of written books")
plt.xlabel("number of books")
plt.ylabel("author name")
plt.show()

##WORDCLOUD GENRE

In [ ]:
print(df['categories'].unique)

In [ ]:
genres = " ".join(
    df["categories"]
    .dropna()
    .str.replace(" ", " ", regex=False)
)

In [ ]:
genres = re.sub(r"<.*?>", "", genres)

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white",
    collocations=False
).generate(genres)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud Book Genre", fontsize=16)
plt.show()

#TEXT CLEANING

##Removing URLs and HTML tags or punctuations from description

In [ ]:
import regex as re
non_latin_pattern = r"[^\x00-\x7FÀ-ÖØ-öø-ÿ]"
mask = ~df["title"].astype(str).str.contains(non_latin_pattern, regex=True)
df = df[mask].reset_index(drop=True)

In [ ]:
url_pattern = re.compile(r'https?://\S+|www\.\S+')
def remove_url(text):
    return re.sub(url_pattern, r'', text)
html_pattern = re.compile('<[^>]*>')
def clean_html_tags(text):
    return re.sub(html_pattern, r'', text)
punctuations = string.punctuation
def remove_punctuations(text):
    return text.translate(str.maketrans('', '', punctuations))

df['description'] = df['description'].astype(str).apply(remove_url)
df['description'] = df['description'].astype(str).apply(clean_html_tags)
df['description'] = df['description'].astype(str).apply(remove_punctuations)

##Convert Letter Case to Lower Case & Clip Extra Space

In [ ]:
df[["title", "authors", "categories", "description"]] = pd.concat([df[col].astype(str).str.lower().str.strip()
                                                                             for col in ["title", "authors", "categories", "description"]],
                                                                            axis=1)

##Remove Book Series Info from Data

In [ ]:
series_pattern = r"(?:[;]\s*|\(\s*)([^\(;]*\s*#\s*\d+(?:\.?\d+|\\&\d+|-?\d*))"
def get_book_series_info(text):
    series_info = re.findall(series_pattern, text)
    if series_info:
        series_info = " ".join([i.replace(" ", "_") for i in series_info])
        return series_info
    else:
        return np.nan
df['BookSeriesInfo'] = df['title'].astype(str).apply(get_book_series_info)

In [ ]:
import re
raw_pattern = r"(?:[\(]\s*[^\(;]*\s*#\s*\d+(?:\.?\d+|\\&\d+|-?\d*)(?:;|\))|\s*[^\(;]*\s*#\s*\d+(?:\.?\d+|\\&\d+|-?\d*)\))"
def clean_title_safe(text):
    if not isinstance(text, str):
        return ""
    return re.sub(raw_pattern, '', text).strip()
df["title"] = df["title"].astype(str).apply(clean_title_safe)

In [ ]:
df["authors"] = df["authors"].str.replace('"','')

##Drop Varians of the Same Book

In [ ]:
df = df.sort_values(by="authors", na_position='last')\
.drop_duplicates(subset=["title", "authors", "description"], keep='first')

In [ ]:
df.head()

In [ ]:
df=df[['title','authors','categories','description','thumbnail','BookSeriesInfo']]

##Clean Multi Values (Split based on comma on authors and categories)

In [ ]:
def clean_multi_values(text):
    if not isinstance(text, str):
        return ""
    items = [i.strip() for i in text.split(',')]
    return " ".join([i.replace(" ", "_") for i in items])
df["authors"] = df["authors"].apply(clean_multi_values)
df["categories"] = df["categories"].apply(clean_multi_values)

#TRANSFORM

In [ ]:
df.head()

##Merge text onto single column

In [ ]:
df["bow"] = df[['title', 'BookSeriesInfo', 'authors', 'categories', 'description']].fillna('').agg(' '.join, axis=1)

In [ ]:
df = df[['title', 'BookSeriesInfo', 'authors', 'categories', 'description', 'thumbnail']]

##Extract Keyword

In [ ]:
kw_model = KeyBERT()
combined_text = df['title'].fillna('') + " " + df['categories'].fillna('') + " " + df['description'].fillna('')
all_texts = combined_text.tolist()
keywords_list = kw_model.extract_keywords(all_texts, keyphrase_ngram_range=(1, 1), stop_words="english")
df["keywords"] = [" ".join([k[0] for k in kw]) for kw in keywords_list]

In [ ]:
df.info()

##Remove duplicate book name

In [ ]:
df = df.drop_duplicates(subset=["title"], keep='first')

#Modelling

##Vectorize Keywords (TF-IDF + ST)

In [ ]:
tfidf_config = {
    'analyzer': 'word',
    'min_df': 3,
    'max_df': 0.6,
    'stop_words': 'english',
    'encoding': 'utf-8',
    'token_pattern': r"(?u)\S\S+"
}
tfidf_author = TfidfVectorizer(**tfidf_config)
tfidf_cat = TfidfVectorizer(**tfidf_config)
tfidf_title = TfidfVectorizer(**tfidf_config)
tfidf_kw = TfidfVectorizer(**tfidf_config)
X_author = tfidf_author.fit_transform(df['authors'].fillna('').str.replace(' ', '_'))
X_cat = tfidf_cat.fit_transform(df['categories'].fillna('').str.replace(' ', '_'))
X_title = tfidf_title.fit_transform(df['title'].fillna(''))
X_kw = tfidf_kw.fit_transform(df['keywords'].fillna(''))

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
X_content_embeddings = model.encode(df['keywords'].fillna('').tolist(), show_progress_bar=True)

In [ ]:
model.save('/your folder path')

In [ ]:
np.save('/your folder path', X_content_embeddings)

#RECS SYSTEM

In [ ]:
import joblib
import pandas as pd

# load data
tfidf_author = joblib.load("tfidf_author.joblib")
tfidf_cat = joblib.load("/tfidf_cat.joblib")
tfidf_title = joblib.load("/tfidf_title.joblib")
tfidf_kw = joblib.load("/tfidf_kw.joblib")

X_author = joblib.load("/X_author.joblib")
X_cat = joblib.load("/X_cat.joblib")
X_title = joblib.load("/X_title.joblib")
X_kw = joblib.load("/X_kw.joblib")

# dataframe + metadata
df = pd.read_pickle("/metadata.pkl")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_perfected(query, n=5):
    query = str(query)
    # transform query
    q_author = tfidf_author.transform([query])
    q_cat = tfidf_cat.transform([query])
    q_title = tfidf_title.transform([query])
    q_kw = tfidf_kw.transform([query])
    # similarity
    sim_author = cosine_similarity(q_author, X_author).flatten()
    sim_cat = cosine_similarity(q_cat, X_cat).flatten()
    sim_title = cosine_similarity(q_title, X_title).flatten()
    sim_kw = cosine_similarity(q_kw, X_kw).flatten()
    # i use weight to make a more personalized recs, you can change the number
    final_score = (0.3 * sim_author) + (0.2 * sim_title) + (0.2 * sim_cat) + (0.3 * sim_kw)
    # ranking
    top_idx = final_score.argsort()[-n:][::-1]
    return df.iloc[top_idx][['title', 'authors', 'categories']]



---



##TESTING

Series Info

In [ ]:
# Recommendations with series information
print("\033[1m{}\033[0m".format("Recommendation (Series Information) based on the read: "))
display(recommend_perfected("carl_sagan cosmos", 5))

In [ ]:
display(recommend_perfected('wheel of time', 5))

In [ ]:
display(recommend_perfected('tolkien', 5))

In [ ]:
display(recommend_perfected('cosmos sagan', 10))

In [ ]:
display(recommend_perfected('Walsh Family #3', 5))

In [ ]:
display(recommend_perfected('stephen_king fairy tale', 10))

Other Book in a Series

In [ ]:
# Recommendations with series information numbered
print("\n\033[1m{}\033[0m".format("Recommendation (Numbered Series) based on the read: The Majolica Murders (Antique Lover, #5)"))
display(recommend_perfected("the majolica murders (antique lover, #5)", 5))

Theme/Genre

In [ ]:
print("\n\033[1m{}\033[0m".format("Recommendation (Theme: Programming) based on the read: The Practice of Programming (Addison-Wesley Professional Computing Series)"))
display(recommend_perfected('the practice of programming (addison-wesley professional computing series)', 5))

Author

In [ ]:
print("\n\033[1m{}\033[0m".format("Recommendation (Author: Dean Koontz) based on the read: Cold Fire"))
display(recommend_perfected("cold fire",5))